# Feature Engineering (Javi)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Cargar los archivos de entrenamiento y prueba
df_train = pd.read_csv('data/hoteles-entrena.csv')
df_pred = pd.read_csv('data/hoteles-prueba.csv')

## Conversión de fechas y extracción de características temporales

In [3]:
df_train['arrival_date'] = pd.to_datetime(df_train['arrival_date'], format='%Y-%m-%d')
df_train['arrival_year'] = df_train['arrival_date'].dt.year
df_train['arrival_month'] = df_train['arrival_date'].dt.month
df_train['arrival_day'] = df_train['arrival_date'].dt.day
df_train['day_of_year'] = df_train['arrival_date'].dt.dayofyear
df_train['arrival_month_sin'] = np.sin(2 * np.pi * df_train['arrival_month'] / 12)
df_train['arrival_month_cos'] = np.cos(2 * np.pi * df_train['arrival_month'] / 12)
df_train['day_of_year_sin'] = np.sin(2 * np.pi * df_train['day_of_year'] / 365)
df_train['day_of_year_cos'] = np.cos(2 * np.pi * df_train['day_of_year'] / 365)


In [4]:
# Aplicar las mismas transformaciones de fecha al conjunto de prueba
df_pred['arrival_date'] = pd.to_datetime(df_pred['arrival_date'], format='%Y-%m-%d')
df_pred['arrival_year'] = df_pred['arrival_date'].dt.year
df_pred['arrival_month'] = df_pred['arrival_date'].dt.month
df_pred['arrival_day'] = df_pred['arrival_date'].dt.day
df_pred['day_of_year'] = df_pred['arrival_date'].dt.dayofyear
df_pred['arrival_month_sin'] = np.sin(2 * np.pi * df_pred['arrival_month'] / 12)
df_pred['arrival_month_cos'] = np.cos(2 * np.pi * df_pred['arrival_month'] / 12)
df_pred['day_of_year_sin'] = np.sin(2 * np.pi * df_pred['day_of_year'] / 365)
df_pred['day_of_year_cos'] = np.cos(2 * np.pi * df_pred['day_of_year'] / 365)


In [5]:
## print columns 
print(df_train.columns)
print(df_pred.columns)


Index(['hotel', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
       'adults', 'children', 'meal', 'country', 'market_segment',
       'distribution_channel', 'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type',
       'average_daily_rate', 'required_car_parking_spaces',
       'total_of_special_requests', 'arrival_date', 'arrival_year',
       'arrival_month', 'arrival_day', 'day_of_year', 'arrival_month_sin',
       'arrival_month_cos', 'day_of_year_sin', 'day_of_year_cos'],
      dtype='object')
Index(['id', 'hotel', 'lead_time', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'meal', 'country', 'market_segment',
       'distribution_channel', 'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       '

## Creación de nuevas características basadas en la estadía

In [6]:
df_train['total_nights'] = df_train['stays_in_weekend_nights'] + df_train['stays_in_week_nights']
df_train['stay_days'] = pd.Series('both', index=df_train.index, dtype='object')
df_train.loc[(df_train['stays_in_weekend_nights'] > 0) & (df_train['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
df_train.loc[(df_train['stays_in_weekend_nights'] == 0) & (df_train['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'
df_train['stays_in_weekend'] = (df_train['stays_in_weekend_nights'] > 0).astype(int)
df_train['weekday'] = df_train['arrival_date'].dt.weekday


In [7]:
# Crear la columna 'total_nights' como la suma de noches de semana y fines de semana
df_pred['total_nights'] = df_pred['stays_in_weekend_nights'] + df_pred['stays_in_week_nights']

# Crear la columna 'stay_days' y asignar los valores 'both', 'weekend' o 'weekdays'
df_pred['stay_days'] = pd.Series('both', index=df_pred.index, dtype='object')
df_pred.loc[(df_pred['stays_in_weekend_nights'] > 0) & (df_pred['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
df_pred.loc[(df_pred['stays_in_weekend_nights'] == 0) & (df_pred['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Crear una columna binaria 'stays_in_weekend' que indica si se quedan al menos una noche en el fin de semana
df_pred['stays_in_weekend'] = (df_pred['stays_in_weekend_nights'] > 0).astype(int)

# Crear la columna 'weekday' que representa el día de la semana en que llega el huésped
df_pred['weekday'] = df_pred['arrival_date'].dt.weekday


## Manejo de valores faltantes

In [8]:
df_train['country'] = df_train['country'].fillna('NON')
df_train['agent'] = df_train['agent'].fillna(0)
df_train['company'] = df_train['company'].fillna(0)


In [9]:
# Llenar los valores faltantes en 'country' con 'NON'
df_pred['country'] = df_pred['country'].fillna('NON')

# Llenar los valores faltantes en 'agent' con 0
df_pred['agent'] = df_pred['agent'].fillna(0)

# Llenar los valores faltantes en 'company' con 0
df_pred['company'] = df_pred['company'].fillna(0)


## Mapeo de valores binarios

In [10]:
df_train['children'] = df_train['children'].map({'children': 1, 'none': 0})
df_train['hotel'] = df_train['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0}).astype(int)
df_train['required_car_parking_spaces'] = df_train['required_car_parking_spaces'].map({'parking': 1, 'none': 0}).astype(int)


In [11]:
## no hay children en el archivo de prueba

# Mapear la columna 'hotel' a valores binarios (1 para 'Resort_Hotel', 0 para 'City_Hotel')
df_pred['hotel'] = df_pred['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0}).astype(int)

# Mapear la columna 'required_car_parking_spaces' a valores binarios (1 para 'parking', 0 para 'none')
df_pred['required_car_parking_spaces'] = df_pred['required_car_parking_spaces'].map({'parking': 1, 'none': 0}).astype(int)

In [12]:
## print columns
print(df_train.columns)
print(df_pred.columns)

Index(['hotel', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
       'adults', 'children', 'meal', 'country', 'market_segment',
       'distribution_channel', 'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type',
       'average_daily_rate', 'required_car_parking_spaces',
       'total_of_special_requests', 'arrival_date', 'arrival_year',
       'arrival_month', 'arrival_day', 'day_of_year', 'arrival_month_sin',
       'arrival_month_cos', 'day_of_year_sin', 'day_of_year_cos',
       'total_nights', 'stay_days', 'stays_in_weekend', 'weekday'],
      dtype='object')
Index(['id', 'hotel', 'lead_time', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'meal', 'country', 'market_segment',
       'distribution_channel', 'is_repeated_guest', 'previous_cancellations',
    

In [13]:
# Crear la columna 'total_nights' en df_train
df_train['total_nights'] = df_train['stays_in_weekend_nights'] + df_train['stays_in_week_nights']

# Crear la columna 'total_nights' en df_pred
df_pred['total_nights'] = df_pred['stays_in_weekend_nights'] + df_pred['stays_in_week_nights']


## Escalado con Min-Max Scaling

In [14]:
# Definir el Min-Max Scaler y las columnas que se van a escalar
scaler = MinMaxScaler()
cols_min_max = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 
                'adults', 'previous_cancellations', 'previous_bookings_not_canceled', 
                'booking_changes', 'days_in_waiting_list', 'average_daily_rate', 'total_nights']

# Aplicar Min-Max Scaling al conjunto de entrenamiento
df_train[cols_min_max] = scaler.fit_transform(df_train[cols_min_max])

# Aplicar el mismo Min-Max Scaling al conjunto de prueba
df_pred[cols_min_max] = scaler.transform(df_pred[cols_min_max])


## One-Hot Encoding para variables categóricas

In [15]:
# Verificar si 'stay_days' está en el conjunto de entrenamiento
if 'stay_days' not in df_train.columns:
    print("'stay_days' no está presente en el conjunto de entrenamiento. Creando la columna...")
    df_train['stay_days'] = pd.Series('both', index=df_train.index, dtype='object')
    df_train.loc[(df_train['stays_in_weekend_nights'] > 0) & (df_train['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
    df_train.loc[(df_train['stays_in_weekend_nights'] == 0) & (df_train['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Verificar si 'stay_days' está en el conjunto de prueba
if 'stay_days' not in df_pred.columns:
    print("'stay_days' no está presente en el conjunto de prueba. Creando la columna...")
    df_pred['stay_days'] = pd.Series('both', index=df_pred.index, dtype='object')
    df_pred.loc[(df_pred['stays_in_weekend_nights'] > 0) & (df_pred['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
    df_pred.loc[(df_pred['stays_in_weekend_nights'] == 0) & (df_pred['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Definir las columnas categóricas para aplicar One-Hot Encoding
cols_one_hot = ['meal', 'country', 'market_segment', 'distribution_channel', 
                'reserved_room_type', 'assigned_room_type', 'deposit_type', 
                'agent', 'company', 'customer_type', 'stay_days']

# Aplicar One-Hot Encoding al conjunto de entrenamiento
df_train_encoded = pd.get_dummies(df_train, columns=cols_one_hot, drop_first=True)

# Aplicar One-Hot Encoding al conjunto de prueba
df_pred_encoded = pd.get_dummies(df_pred, columns=cols_one_hot, drop_first=True)

# Alinear las columnas del conjunto de prueba con las del conjunto de entrenamiento
missing_cols = set(df_train_encoded.columns) - set(df_pred_encoded.columns)
for col in missing_cols:
    df_pred_encoded[col] = 0  # Agregar las columnas faltantes con valor 0

# Asegurarse de que el conjunto de prueba tenga las mismas columnas que el conjunto de entrenamiento
df_pred_encoded = df_pred_encoded[df_train_encoded.columns]



/var/folders/8v/p9jmdytd6y36kdsq55l4_cqw0000gn/T/ipykernel_39430/2098019991.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pred_encoded[col] = 0  # Agregar las columnas faltantes con valor 0
/var/folders/8v/p9jmdytd6y36kdsq55l4_cqw0000gn/T/ipykernel_39430/2098019991.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pred_encoded[col] = 0  # Agregar las columnas faltantes con valor 0
/var/folders/8v/p9jmdytd6y36kdsq55l4_cqw0000gn/T/ipykernel_39430/2098019991.py:29: PerformanceWarning: DataFrame is highly fragmented.

## Eliminar columnas no necesarias

In [16]:
df_train.drop(columns=['arrival_date'], inplace=True)
df_pred.drop(columns=['arrival_date'], inplace=True)


## Exportar los datos preprocesados

In [27]:
df_train.to_csv('data/hoteles-entrena-limpio-normalizado.csv', index=False)
df_pred.to_csv('data/hoteles-prueba-limpio-normalizado.csv', index=False)


# Adaboost

In [17]:
# Cargar los datos preprocesados
df_train_encoded = pd.read_csv('data/hoteles-entrena-limpio-normalizado.csv')
df_pred_encoded = pd.read_csv('data/hoteles-prueba-limpio-normalizado.csv')

# Separar la variable objetivo (children) de las características
X_train = df_train_encoded.drop(columns=['children'])  # Asegúrate que 'children' esté en el conjunto de entrenamiento
y_train = df_train_encoded['children']


## Modelo 

## Predicciones 